# Lab 3: Decision Making và Actuation Control Logic

Chào mừng bạn đến với Lab 3. Trong phần này, chúng ta sẽ xây dựng lớp điều khiển cuối cùng của hệ thống ADAS: chuyển từ nhận thức (perception) sang hành động (actuation).

---

## Tổng quan

Sau khi hệ thống đã:
- Phát hiện vật thể
- Tính toán khoảng cách
- Lọc ROI

Bước cuối cùng là **ra quyết định điều khiển**

Lab này gồm 3 phần:

1. Emergency Braking System (EBS)
2. Adaptive Cruise Control (PID)
3. Optimization (Deadband + Slew Rate)

---

## Mục tiêu học tập

- Hiểu logic ưu tiên an toàn (safety-first)
- Implement PID controller
- Giảm rung bằng deadband
- Làm mượt điều khiển bằng slew rate

---

In [ ]:
import numpy as np

# 1. Emergency Braking System (EBS)

## Lý thuyết

Nếu khoảng cách <= ngưỡng nguy hiểm → dừng ngay lập tức.

Đây là một interrupt:
- Không qua PID
- Không qua smoothing
- Ưu tiên tuyệt đối

In [ ]:
CMD_STOP = 8

def emergency_brake(distance, threshold=10.0):
    """
    distance: khoảng cách (cm)
    threshold: ngưỡng nguy hiểm
    """

    ### TODO ###
    if distance <= threshold:
        return CMD_STOP
    
    return None

# 2. Adaptive Cruise Control (PID)

## Lý thuyết

Error:
e(t) = D_measured - D_target

Control:
u(t) = Kp * e + Ki * sum(e) + Kd * (e - e_prev)

In [ ]:
class PIDController:
    def __init__(self, Kp, Ki, Kd):
        self.Kp = Kp
        self.Ki = Ki
        self.Kd = Kd
        
        self.integral = 0
        self.prev_error = 0

    def update(self, distance, target, dt=1.0):
        ### TODO ###
        error = distance - target
        
        # tích phân
        self.integral += error * dt
        
        # đạo hàm
        derivative = (error - self.prev_error) / dt[...]
        
        # PID output
        output = (
            self.Kp * error +
            self.Ki * self.integral +
            self.Kd * derivative
        )
        self.prev_error = error
        return output

# 3. Deadband

## Lý thuyết

Nếu lỗi nhỏ (|e| < threshold) → bỏ qua (coi như = 0)

→ tránh rung lắc nhỏ

In [ ]:
def apply_deadband(error, threshold=2.5):
    ### TODO ###
    if abs(error) < threshold:
        return 0.0
    return error

# 4. Slew Rate Limiting

## Lý thuyết

Giới hạn tốc độ thay đổi của output:

delta = output - prev_output

Nếu vượt quá max_delta → clamp lại

In [ ]:
def apply_slew_rate(output, prev_output, max_delta=5.0):
    delta = output - prev_output

    ### TODO ###
    if delta > max_delta:
        output = prev_output + max_delta
    elif delta < -max_delta:
        output = prev_output - max_delta

    return output

# 5. Pipeline tổng hợp

Thứ tự xử lý:

1. Check EBS
2. Nếu không dừng:
   - Tính PID
   - Apply Deadband
   - Apply Slew Rate

In [ ]:
def control_step(distance, target, pid, prev_output):
    
    # 1. EBS
    stop_cmd = emergency_brake(distance)
    if stop_cmd is not None:
        return stop_cmd, 0

    # 2. PID
    raw_output = pid.update(distance, target)

    # TODO: áp dụng deadband
    error = distance - target
    error = apply_deadband(error)

    # TODO: recompute output nếu cần
    output = (
        pid.Kp * error +
        pid.Ki * pid.integral +
        pid.Kd * derivative
    )
    # 3. Slew rate
    output = apply_slew_rate(output, prev_output)

    return 1, output  # 1 = chạy bình thường

In [ ]:
pid = PIDController(Kp=2.0, Ki=0.1, Kd=1.0)

distances = [20, 18, 15, 13, 12, 11, 9]
prev_output = 0

print("Simulation:")

for d in distances:
    cmd, output = control_step(d, target=12.5, pid=pid, prev_output=prev_output)
    
    print(f"Distance: {d} | CMD: {cmd} | Output: {output:.2f}")
    
    prev_output = output